<div style="background: linear-gradient(135deg, #0f0c29, #302b63, #24243e); padding: 36px 32px; border-radius: 16px; margin-bottom: 8px;">
  <h1 style="color: #fff; font-family: 'Georgia', serif; font-size: 2.2em; margin: 0 0 8px 0; letter-spacing: -1px;">🎙️ Week 4: AI Meeting Summarizer</h1>
  <p style="color: #a78bfa; font-size: 1.1em; margin: 0; font-family: monospace;">WAV Upload → Whisper Transcription → LLM Summary → ROUGE Evaluation</p>
  <hr style="border-color: #4c3f8a; margin: 20px 0 16px 0;">
  <table style="color: #e0d7ff; font-size: 0.95em; font-family: monospace; border-collapse: collapse;">
    <tr><td style="padding: 4px 16px 4px 0;">📦 Step 1</td><td>Install dependencies</td></tr>
    <tr><td style="padding: 4px 16px 4px 0;">🔑 Step 2</td><td>Enter Groq API key</td></tr>
    <tr><td style="padding: 4px 16px 4px 0;">🎵 Step 3</td><td><strong style="color:#f0abfc">Upload your .wav file</strong></td></tr>
    <tr><td style="padding: 4px 16px 4px 0;">📝 Step 4</td><td>Transcribe audio (Whisper)</td></tr>
    <tr><td style="padding: 4px 16px 4px 0;">🤖 Step 5</td><td>Generate AI summary</td></tr>
    <tr><td style="padding: 4px 16px 4px 0;">📊 Step 6</td><td>ROUGE evaluation</td></tr>
  </table>
</div>

## 📦 Step 1 — Install Dependencies

In [1]:
print("Installing packages... (takes ~1 min on first run)")
!pip install groq openai-whisper rouge-score pydub ipywidgets --quiet
!apt-get install -y ffmpeg -qq
print("\n✅ All packages ready!")

Installing packages... (takes ~1 min on first run)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 12.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 40.8 MB/s eta 0:00:00

✅ All packages ready!


## 🔑 Step 2 — Enter Groq API Key
> Get a free key at **https://console.groq.com** → API Keys

In [2]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import os

display(HTML("<p style='font-size:1em; color:#555;'>🔑 Paste your Groq API key below:</p>"))

api_key_widget = widgets.Password(
    placeholder='gsk_...',
    layout=widgets.Layout(width='440px', height='38px')
)
display(api_key_widget)
display(HTML("<p style='color:#888; font-size:0.85em; margin-top:6px;'>🔒 Hidden input — key is not displayed</p>"))
display(HTML("<p style='color:#aaa; font-size:0.82em;'>No key? Select <b>HuggingFace BART</b> in Step 5 instead (no key needed)</p>"))

Password(layout=Layout(height='38px', width='440px'), placeholder='gsk_...')

## 🎵 Step 3 — Upload Your .wav File

In [3]:
from google.colab import files
import os

display(HTML("""
<div style='background:#f0f7ff; border:2px dashed #4f8ef7; border-radius:12px;
     padding:24px 28px; margin:8px 0; text-align:center;'>
  <div style='font-size:2.4em; margin-bottom:6px;'>🎙️</div>
  <div style='font-size:1.1em; font-weight:bold; color:#1a56db;'>Click below to upload your meeting audio</div>
  <div style='color:#555; font-size:0.9em; margin-top:4px;'>Supported: .wav &nbsp;|&nbsp; .mp3 &nbsp;|&nbsp; .m4a &nbsp;|&nbsp; .flac</div>
</div>
"""))

uploaded = files.upload()

if uploaded:
    AUDIO_FILE = list(uploaded.keys())[0]
    size_kb = os.path.getsize(AUDIO_FILE) / 1024
    display(HTML(f"""
    <div style='background:#f0fff4; border:1px solid #34d399; border-radius:8px; padding:14px 18px; margin-top:10px;'>
      <b style='color:#065f46;'>✅ File uploaded successfully!</b><br>
      <span style='font-family:monospace; color:#047857;'>📁 {AUDIO_FILE}</span>
      <span style='color:#6b7280; font-size:0.88em;'> ({size_kb:.1f} KB)</span><br>
      <span style='color:#555; font-size:0.9em;'>Ready for transcription → Run Step 4</span>
    </div>
    """))
else:
    AUDIO_FILE = None
    print("⚠️ No file uploaded yet.")

Saving meeting1.wav to meeting1.wav


## 📝 Step 4 — Transcribe with Whisper
> OpenAI Whisper runs **locally** on Colab — no API key needed for transcription.

In [4]:
import whisper
import time

# ── Whisper model size selector ──
display(HTML("<p style='font-size:1em; color:#444; margin-bottom:6px;'>🔧 Choose Whisper model size:</p>"))
whisper_model_widget = widgets.RadioButtons(
    options=[
        ('tiny  — Fastest (lower accuracy)', 'tiny'),
        ('base  — Recommended balance ✅', 'base'),
        ('small — Better accuracy (slower)', 'small'),
        ('medium — Best accuracy (slowest)', 'medium'),
    ],
    value='base',
    layout=widgets.Layout(width='400px')
)
display(whisper_model_widget)

transcribe_btn = widgets.Button(
    description='🎙️ Transcribe Audio',
    button_style='primary',
    layout=widgets.Layout(width='220px', height='44px', margin='12px 0 0 0')
)
transcribe_output = widgets.Output()

TRANSCRIPT_TEXT = ""

def run_transcription(b):
    global TRANSCRIPT_TEXT, AUDIO_FILE
    with transcribe_output:
        clear_output(wait=True)
        if not AUDIO_FILE or not os.path.exists(AUDIO_FILE):
            display(HTML("<p style='color:red;'>❌ No audio file found. Please upload in Step 3 first.</p>"))
            return

        model_size = whisper_model_widget.value
        display(HTML(f"<p>⏳ Loading Whisper <b>{model_size}</b> model (downloads once)...</p>"))

        t0 = time.time()
        model = whisper.load_model(model_size)

        display(HTML(f"<p>🎙️ Transcribing <b>{AUDIO_FILE}</b>...</p>"))
        result = model.transcribe(AUDIO_FILE)
        TRANSCRIPT_TEXT = result["text"].strip()
        elapsed = round(time.time() - t0, 1)

        word_count = len(TRANSCRIPT_TEXT.split())
        display(HTML(f"""
        <div style='background:#f0fff4; border:1px solid #6ee7b7; border-radius:10px; padding:14px 18px; margin-top:8px;'>
          <b style='color:#065f46;'>✅ Transcription complete in {elapsed}s</b><br>
          <span style='color:#047857; font-size:0.9em;'>{word_count} words detected · Language: {result.get('language','unknown')}</span>
        </div>
        """))

        print("\n" + "─"*60)
        print("📄 TRANSCRIPT:")
        print("─"*60)
        print(TRANSCRIPT_TEXT)
        print("─"*60)
        display(HTML("<p style='color:#555; font-size:0.9em; margin-top:8px;'>✅ Transcript ready → Run Step 5 to summarize</p>"))

transcribe_btn.on_click(run_transcription)
display(transcribe_btn)
display(transcribe_output)

RadioButtons(index=1, layout=Layout(width='400px'), options=(('tiny  — Fastest (lower accuracy)', 'tiny'), ('b…

Button(button_style='primary', description='🎙️ Transcribe Audio', layout=Layout(height='44px', margin='12px 0 …

Output()

## 🤖 Step 5 — Generate AI Summary

In [5]:
PROMPT_TEMPLATES = {
    "📋 Full Summary (Overview + Key Points + Decisions + Action Items)": """You are a professional meeting summarizer. Analyze the transcript and return a structured summary in EXACTLY this format:

**MEETING SUMMARY**

**Overview:**
[2-3 sentence overview]

**Key Points:**
- [point]

**Decisions Made:**
- [decision]

**Action Items:**
| Owner | Task | Deadline |
|-------|------|----------|
| [name] | [task] | [deadline or TBD] |

**Next Steps:**
- [step]

TRANSCRIPT:
{transcript}""",

    "🔑 Key Points Only": """Extract the KEY POINTS from this meeting transcript as a clear bullet list.

**Key Points:**
- [point]

TRANSCRIPT:
{transcript}""",

    "✅ Action Items Only": """Extract all ACTION ITEMS from this meeting transcript.

| Owner | Task | Deadline |
|-------|------|----------|
| [name] | [task] | [deadline] |

TRANSCRIPT:
{transcript}""",

    "⚖️ Decisions Only": """List all DECISIONS made in this meeting transcript.

**Decisions Made:**
- [decision]

TRANSCRIPT:
{transcript}""",
}

# ── Backend selector ──
display(HTML("<p style='font-size:1em; color:#444; margin-bottom:4px;'><b>🤖 Choose AI backend:</b></p>"))
backend_widget = widgets.RadioButtons(
    options=[
        ('🚀 Groq — LLaMA 3.1 (fast, needs API key)', 'groq'),
        ('🤗 HuggingFace — BART (local, no key needed)', 'huggingface'),
    ],
    value='groq',
    layout=widgets.Layout(width='440px')
)
display(backend_widget)

# ── Summary type selector ──
display(HTML("<p style='font-size:1em; color:#444; margin:12px 0 4px 0;'><b>📋 Choose summary type:</b></p>"))
summary_type_widget = widgets.RadioButtons(
    options=list(PROMPT_TEMPLATES.keys()),
    value=list(PROMPT_TEMPLATES.keys())[0],
    layout=widgets.Layout(width='600px')
)
display(summary_type_widget)

# ── Manual transcript override ──
display(HTML("<p style='font-size:0.9em; color:#666; margin:12px 0 4px 0;'>✏️ Transcript preview (auto-filled from Step 4, or edit here):</p>"))
transcript_edit_widget = widgets.Textarea(
    value='',
    placeholder='Transcript will appear here after Step 4 transcription. Or paste your own text manually...',
    layout=widgets.Layout(width='100%', height='160px')
)
display(transcript_edit_widget)

refresh_btn = widgets.Button(description='🔄 Load from Step 4', button_style='warning',
                              layout=widgets.Layout(width='200px', margin='6px 0 0 0'))
def refresh_transcript(b):
    transcript_edit_widget.value = TRANSCRIPT_TEXT
    with summarize_output:
        clear_output()
        if TRANSCRIPT_TEXT:
            display(HTML("<p style='color:green;'>✅ Transcript loaded from Step 4.</p>"))
        else:
            display(HTML("<p style='color:orange;'>⚠️ No transcript yet — run Step 4 first (or paste manually above).</p>"))
refresh_btn.on_click(refresh_transcript)
display(refresh_btn)

# ── Summarize button ──
summarize_btn = widgets.Button(
    description='⚡ Generate Summary',
    button_style='success',
    layout=widgets.Layout(width='220px', height='44px', margin='14px 0 0 0')
)
summarize_output = widgets.Output()
LAST_SUMMARY = ""

def generate_summary(b):
    global LAST_SUMMARY
    with summarize_output:
        clear_output(wait=True)
        transcript = transcript_edit_widget.value.strip()
        if not transcript:
            display(HTML("<p style='color:red;'>❌ No transcript found. Run Step 4 or paste text manually above.</p>"))
            return

        backend = backend_widget.value
        summary_type = summary_type_widget.value
        prompt = PROMPT_TEMPLATES[summary_type].format(transcript=transcript)

        if backend == 'groq':
            api_key = api_key_widget.value.strip()
            if not api_key:
                display(HTML("<p style='color:red;'>❌ Please enter your Groq API key in Step 2.</p>"))
                return
            display(HTML(f"<p>🚀 Calling Groq LLaMA 3.1 for <b>{summary_type}</b>...</p>"))
            try:
                from groq import Groq
                client = Groq(api_key=api_key)
                response = client.chat.completions.create(
                    model='llama-3.1-8b-instant',
                    messages=[{'role': 'user', 'content': prompt}],
                    temperature=0.3,
                    max_tokens=1500
                )
                LAST_SUMMARY = response.choices[0].message.content
            except Exception as e:
                display(HTML(f"<p style='color:red;'>❌ Groq error: {e}</p>"))
                return

        else:
            display(HTML("<p>🤗 Loading HuggingFace BART model (first run downloads ~1.6GB)...</p>"))
            try:
                from transformers import pipeline
                summarizer = pipeline('summarization', model='facebook/bart-large-cnn')
                text = transcript[:3000]
                result = summarizer(text, max_length=300, min_length=60, do_sample=False)
                LAST_SUMMARY = result[0]['summary_text']
            except Exception as e:
                display(HTML(f"<p style='color:red;'>❌ HuggingFace error: {e}</p>"))
                return

        display(HTML("""
        <div style='background:#f0fff4; border:1px solid #34d399; border-radius:10px;
             padding:12px 18px; margin:8px 0; color:#065f46; font-weight:bold;'>
          ✅ Summary generated successfully!
        </div>
        """))
        print("\n" + "═"*62)
        print(f"  {summary_type}")
        print("═"*62)
        print(LAST_SUMMARY)
        print("═"*62)
        display(HTML("<p style='color:#555; font-size:0.9em; margin-top:8px;'>✅ Done! Scroll down to Step 6 for ROUGE evaluation or Step 7 to save.</p>"))

summarize_btn.on_click(generate_summary)
display(summarize_btn)
display(summarize_output)

# Simulate button click to generate summary
generate_summary(None)

RadioButtons(layout=Layout(width='440px'), options=(('🚀 Groq — LLaMA 3.1 (fast, needs API key)', 'groq'), ('🤗 …

RadioButtons(layout=Layout(width='600px'), options=('📋 Full Summary (Overview + Key Points + Decisions + Actio…

Textarea(value='', layout=Layout(height='160px', width='100%'), placeholder='Transcript will appear here after…

Button(button_style='warning', description='🔄 Load from Step 4', layout=Layout(margin='6px 0 0 0', width='200p…

Button(button_style='success', description='⚡ Generate Summary', layout=Layout(height='44px', margin='14px 0 0…

Output()

## 📊 Step 6 — ROUGE Evaluation
> Paste a **human reference summary** to score the AI output against it.

In [7]:
from rouge_score import rouge_scorer as rs

display(HTML("<p style='font-size:1em; color:#444; margin-bottom:6px;'>📝 Paste a human-written reference summary (ground truth):</p>"))

reference_widget = widgets.Textarea(
    placeholder='Write or paste a reference summary here for comparison...',
    layout=widgets.Layout(width='100%', height='130px')
)
display(reference_widget)

rouge_btn = widgets.Button(
    description='📊 Calculate ROUGE Scores',
    button_style='info',
    layout=widgets.Layout(width='250px', height='44px', margin='10px 0 0 0')
)
rouge_output = widgets.Output()

def run_rouge(b):
    with rouge_output:
        clear_output(wait=True)
        global LAST_SUMMARY
        if not LAST_SUMMARY:
            display(HTML("<p style='color:red;'>❌ No summary yet — generate one in Step 5 first.</p>"))
            return
        reference = reference_widget.value.strip()
        if not reference:
            display(HTML("<p style='color:orange;'>⚠️ Please enter a reference summary above.</p>"))
            return

        scorer = rs.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
        scores = scorer.score(reference, LAST_SUMMARY)

        print("\n" + "═"*54)
        print("   📊 ROUGE EVALUATION RESULTS")
        print("═"*54)

        results = [
            ("ROUGE-1", scores['rouge1'],  0.60, "Word overlap"),
            ("ROUGE-2", scores['rouge2'],  0.35, "Phrase overlap"),
            ("ROUGE-L", scores['rougeL'],  0.50, "Sequence match"),
        ]
        f1_scores = []
        for name, score, threshold, desc in results:
            p  = round(score.precision, 3)
            r  = round(score.recall,    3)
            f1 = round(score.fmeasure,  3)
            f1_scores.append(f1)
            status = "✅ PASS" if f1 >= threshold else "⚠️  LOW"
            filled = int(f1 * 28)
            bar = "█" * filled + "░" * (28 - filled)
            print(f"\n  {name}  [{bar}]  F1 = {f1}")
            print(f"  {desc}  |  P={p}  R={r}  |  Threshold ≥{threshold}  {status}")

        avg = round(sum(f1_scores)/3, 3)
        print("\n" + "─"*54)
        print(f"  Average F1:  {avg}")
        if avg >= 0.50:
            print("  🏆 Overall: GOOD — strong summary quality!")
        elif avg >= 0.35:
            print("  🔶 Overall: ACCEPTABLE — try refining the prompt")
        else:
            print("  🔴 Overall: LOW — try a larger model or better prompt")
        print("═"*54)

rouge_btn.on_click(run_rouge)
display(rouge_btn)
display(rouge_output)

Textarea(value='', layout=Layout(height='130px', width='100%'), placeholder='Write or paste a reference summar…

Button(button_style='info', description='📊 Calculate ROUGE Scores', layout=Layout(height='44px', margin='10px …

Output()

## 💾 Step 7 — Save Summary

In [8]:
import json
from datetime import datetime
from google.colab import files as colab_files

fname_widget = widgets.Text(
    value='meeting_summary',
    description='Filename:',
    layout=widgets.Layout(width='300px')
)
fmt_widget = widgets.Dropdown(
    options=[('.txt — Plain text', 'txt'), ('.md — Markdown', 'md'), ('.json — JSON with metadata', 'json')],
    value='txt',
    layout=widgets.Layout(width='260px')
)
save_btn = widgets.Button(
    description='💾 Download Summary',
    button_style='primary',
    layout=widgets.Layout(width='210px', height='42px', margin='10px 0 0 0')
)
save_output = widgets.Output()

def save_and_download(b):
    with save_output:
        clear_output(wait=True)
        global LAST_SUMMARY, AUDIO_FILE
        if not LAST_SUMMARY:
            display(HTML("<p style='color:red;'>❌ No summary to save. Run Step 5 first.</p>"))
            return

        fmt  = fmt_widget.value.split()[0].replace('.', '')
        name = fname_widget.value.strip() or 'meeting_summary'
        full = f"{name}.{fmt}"
        ts   = datetime.now().strftime("%Y-%m-%d %H:%M")

        if fmt == 'json':
            data = {
                "timestamp": ts,
                "audio_file": AUDIO_FILE or "unknown",
                "backend": backend_widget.value,
                "summary_type": summary_type_widget.value,
                "transcript": transcript_edit_widget.value,
                "summary": LAST_SUMMARY
            }
            with open(full, 'w') as f:
                json.dump(data, f, indent=2)
        else:
            header = f"Meeting Summary — {ts}\nAudio: {AUDIO_FILE or 'n/a'}\n{'='*60}\n\n"
            with open(full, 'w') as f:
                f.write(header + LAST_SUMMARY)

        display(HTML(f"<p style='color:green;'>✅ Saving <b>{full}</b>...</p>"))
        colab_files.download(full)

save_btn.on_click(save_and_download)
display(widgets.HBox([fname_widget, fmt_widget]))
display(save_btn)
display(save_output)

Button(button_style='primary', description='💾 Download Summary', layout=Layout(height='42px', margin='10px 0 0…

Output()

---
<div style='background:#1e1e2e; border-radius:12px; padding:20px 24px; color:#cdd6f4;'>
<h3 style='color:#cba6f7; margin-top:0;'>📐 Pipeline Architecture</h3>
<pre style='color:#a6e3a1; background:transparent; margin:0;'>
.wav / .mp3 / .flac
       │
       ▼
  Whisper ASR  ──────────────────────────────  local, free
       │
       ▼
  Raw Transcript
       │
       ├──► Groq (LLaMA 3.1-8b)  ──────────  cloud, API key
       └──► HuggingFace BART     ──────────  local, no key
       │
       ▼
  Structured Summary
  (Overview · Key Points · Decisions · Actions)
       │
       ▼
  ROUGE-1 / ROUGE-2 / ROUGE-L Evaluation
       │
       ▼
  Download as .txt / .md / .json
</pre>
</div>